# Side panel for a running Colab A/B
Open this while the run notebook is busy and select the **same Colab server** (Select Kernel > Colab > the
server already in use, not a new one). It runs in its own kernel on the same VM. Set `TAG` to the run's tag.


In [ ]:
TAG = '31b_ca5lora_20261123'
import shutil, subprocess
from pathlib import Path
running = subprocess.run("pgrep -f live_status.py", shell=True, capture_output=True, text=True).stdout.split()
if running:
    print('live status already running, pid', running)
else:
    src = Path(f'/content/drive/MyDrive/gemma_fasteval/launch/{TAG}/_live_status.py')
    shutil.copyfile(src, '/content/live_status.py')
    subprocess.Popen(f'nohup /content/venv/bin/python /content/live_status.py {TAG} 120 > /content/live_status.log 2>&1 &',
                     shell=True)
    print('live status started; Drive ab_runs/' + TAG + '/live_status.txt every 2 min')

In [ ]:
# current status: driver, per-arm progress, vLLM speed, log tail (returns immediately; rerun any time)
import json, time
from pathlib import Path
live = Path(f'/content/drive/MyDrive/gemma_fasteval/ab_runs/{TAG}/live_status.txt')
print(live.read_text() if live.exists() else 'live status not written yet')
prog = Path('/content/ab/progress.json')
print('passes on record:', [(p['arm'], p['round'], p['solved']) for p in json.loads(prog.read_text())['passes']]
      if prog.exists() else 'none yet')

In [ ]:
# any shell command on the VM, e.g. tail of the driver log or GPU state
!tail -n 5 /content/ab.log; nvidia-smi --query-gpu=utilization.gpu,memory.used --format=csv,noheader